In [1]:
import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """
    Find the project root directory by looking for a `pyproject.toml` file.

    Args:
        start (Path | None): Optional start path

    Returns:
        Path: The project root directory

    Raises:
        RuntimeError: If the project root cannot be found
    """
    current = (start or Path.cwd()).resolve()

    for parent in [current, *current.parents]:
        if (parent / "pyproject.toml").exists():
            return parent

    raise RuntimeError("Could not locate project root")


PROJECT_ROOT = find_project_root()

SOURCE_DIR = PROJECT_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SOURCE_DIR}")

Project root: /Users/brenoingwersensantos/Documents/GitHub/vrptw-lab
Source directory: /Users/brenoingwersensantos/Documents/GitHub/vrptw-lab/src


In [ ]:
from pathlib import Path

import pandas as pd

In [3]:
def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize the column names to snake case and remove periods.
    """
    df.columns = [
        col.strip().lower().replace(" ", "_").replace(".", "") for col in df.columns
    ]
    return df


def load_solomon_data(path: Path):
    """
    Load Solomon data from a CSV file.
    """
    df = _normalize_columns(pd.read_csv(path))
    return df

In [4]:
name = "solomon"
instance = "C1/C101.csv"
max_trucks = 20
truck_capacity = 300

In [5]:
path = Path(f"../data/{name}/{instance}")
df = load_solomon_data(path)
print(df.shape)
df.head()

(101, 7)


,cust_no,xcoord,ycoord,demand,ready_time,due_date,service_time
0,1,40,50,0,0,1236,0
1,2,45,68,10,912,967,90
2,3,45,70,30,825,870,90
3,4,42,66,10,65,146,90
4,5,42,68,10,727,782,90


In [6]:
from vrptw.instance import VRPTWInstance
from vrptw.solver import VRPTWSolver

instance = VRPTWInstance.from_df(df)
solver = VRPTWSolver(instance, max_trucks=max_trucks, truck_capacity=truck_capacity)

2026-09-12 20:35:36.379 | INFO     | vrptw.solver:_build_problem:70 - Building the VRPTW problem...
2026-09-12 20:35:36.379 | INFO     | vrptw.solver:_add_variables:78 - Adding the decision variables to the model...
2026-09-12 20:35:36.538 | INFO     | vrptw.solver:_add_variables:88 - Arc variables: (10100,)
2026-09-12 20:35:36.538 | INFO     | vrptw.solver:_add_variables:89 - Node load variables: (101,)
2026-09-12 20:35:36.539 | INFO     | vrptw.solver:_add_constraints:95 - Adding constraints to the model...
2026-09-12 20:35:36.539 | INFO     | vrptw.solver:_add_circuit_constraint:112 - Constraint: ensure that each node is visited (multiple circuits allowed).
2026-09-12 20:35:36.548 | INFO     | vrptw.solver:_add_max_trucks_constraint:124 - Constraint: limit the number of trucks to 20.
2026-09-12 20:35:36.549 | INFO     | vrptw.solver:_add_load_constraint:131 - Constraint: limit the load on each truck route.


In [7]:
result = solver.solve(max_time_in_seconds=30)

2026-09-12 20:35:36.593 | INFO     | vrptw.solver:solve:257 - Solving the VRPTW problem...
2026-09-12 20:35:36.593 | INFO     | vrptw.solver:_solve_stage1_minimize_trucks:224 - Stage 1: minimizing trucks...
2026-09-12 20:35:36.594 | INFO     | vrptw.solver:_add_truck_minimization_objective:155 - Objective: minimize the number of trucks used
2026-09-12 20:35:38.442 | INFO     | vrptw.callback:on_solution_callback:17 - Solution found with objective: 7.0
2026-09-12 20:35:38.675 | SUCCESS  | vrptw.solver:_build_solve_result:200 - Solver finished with status: OPTIMAL


In [8]:
print(f"Status: {result.status_name}")
print(f"Trucks: {result.n_trucks}")
print(f"Runtime: {result.runtime_seconds:.2f}s")
print(f"Solver status: {solver.status_name}")

Status: OPTIMAL
Trucks: 7
Runtime: 2.07s
Solver status: OPTIMAL


In [9]:
stops_df = result.solution.build_stops_df()
stops_df

,truck_id,sequence,cust_no_from,cust_no_to
0,1,1,1,10
1,1,2,10,74
2,1,3,74,57
3,1,4,57,53
4,1,5,53,32
...,...,...,...,...
102,7,13,45,80
103,7,14,80,6
104,7,15,6,67
105,7,16,67,13


In [10]:
stops_df = pd.merge(
    stops_df,
    df[["cust_no", "demand"]],
    left_on="cust_no_to",
    right_on="cust_no",
    how="left",
    validate="many_to_one"
).drop(columns=["cust_no"])
stops_df

,truck_id,sequence,cust_no_from,cust_no_to,demand
0,1,1,1,10,10
1,1,2,10,74,10
2,1,3,74,57,30
3,1,4,57,53,10
4,1,5,53,32,20
...,...,...,...,...,...
102,7,13,45,80,10
103,7,14,80,6,10
104,7,15,6,67,10
105,7,16,67,13,20


In [11]:
stops_df.groupby("truck_id").agg(
    total_stops=("sequence", "max"),
    start_from=("cust_no_from", "first"),
    end_at=("cust_no_to", "last"),
    total_demand=("demand", "sum"),
)

,total_stops,start_from,end_at,total_demand
truck_id,,,,
1,11,1,1,180
2,13,1,1,250
3,13,1,1,270
4,16,1,1,300
5,18,1,1,260
6,19,1,1,300
7,17,1,1,250
